# Swiss Commutes — inspect the audit

## tl;dr
The statistical totals largely reproduce, but their interpretation does not. The audit records 16 findings and 17 failed check instances among 335 checks. Resident mode shares are reused for inbound commuters; the same OD pair gets different mode totals across city pages. Geneva mixes scopes and assigns “no transport” to walking/cycling.

This notebook reads the full saved evidence. The HTML report contains the charts and recommendations. Change `CITY` below to investigate another city.

## Context & Methods
The unit is a city-page origin–destination–mode–direction row, measured in estimated workers. These are not vehicles or observed daily trips. Never sum city pages as unique national people. Source years differ (2020 OD, 2023 shares, 2025 border totals, 2026 routing inputs).

### Key Assumptions
Five-minute outputs are simulations. Route endpoints and the population curve use different arrival definitions. The 2019–2021 benchmark is a sensitivity reference, not a replacement estimate. The original transformations are in `scripts/audit-data.mjs`, `audit-sources.py`, `audit-insee.py`; hashes are in `audit.json`.

The code cells below were run top-to-bottom with Python 3 and outputs captured. No Jupyter kernel was installed in the authoring environment. Only the standard library is required by these cells.

In [1]:
from pathlib import Path
import json, csv, hashlib, subprocess
from pprint import pprint

CITY = "zurich"
RECOMPUTE = False  # True reruns the Node/Python audit; no network or routing requests.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app/cities.ts").exists())
folder = root / "audits/2026-09-06"
if RECOMPUTE:
    run = subprocess.run(["node", "--experimental-strip-types", "scripts/audit-data.mjs"], cwd=root, capture_output=True, text=True, check=True)
audit = json.loads((folder / "audit.json").read_text())
assert CITY in {r["city"] for r in audit["cityRows"]}
print("Snapshot:", audit["generatedAt"], "City:", CITY)
print("Cities:", len(audit["cityRows"]), "Corridor rows:", len(audit["routes"]))

Snapshot: 2026-09-06T17:45:32.847Z City: zurich
Cities: 12 Corridor rows: 18834


## Data
### Verify that the inspected application files still match the snapshot
A mismatch means the report is stale for that file. Recompute the audit and report before interpreting it as current.

In [2]:
changed = []
for row in audit["manifest"]:
    file = root / row["file"]
    if not file.exists() or hashlib.sha256(file.read_bytes()).hexdigest() != row["sha256"]:
        changed.append(row["file"])
print("Changed or missing application/script inputs:", changed)
assert not changed, "Inputs changed: rerun the audit before using current-state conclusions"
for row in audit["originalSources"]["files"]:
    file = folder / "sources" / row["file"]
    assert hashlib.sha256(file.read_bytes()).hexdigest() == row["sha256"], row["file"]
print("Saved public-source hashes match.")

Changed or missing application/script inputs: []
Saved public-source hashes match.


## Results
### 1. See every failed check
These instances overlap: the 14 pair discrepancies share one allocation defect; the Geneva checks share a scope problem. The endpoint threshold is a review flag.

In [3]:
failed = [r for r in audit["checks"] if r["status"] == "fail"]
print("Passed:", len(audit["checks"]) - len(failed), "Failed instances:", len(failed))
for r in failed:
    print(r["city"], "|", r["check"], "|", r["actual"], "vs", r["expected"])

Passed: 318 Failed instances: 17
zurich | Route endpoint snap within review threshold | 1 vs 0
geneva | swissOutbound population scope matches summary | 8703 vs 6881
geneva | Full-canton inbound population versus OCSTAT | 23398 vs 25628
zurich → winterthur | Same OD car allocation on both city pages | 553 vs 1040
zurich → lucerne | Same OD car allocation on both city pages | 209 vs 363
bern → biel-bienne | Same OD car allocation on both city pages | 96 vs 194
bern → la-chaux-de-fonds | Same OD car allocation on both city pages | 1 vs 4
winterthur → zurich | Same OD car allocation on both city pages | 3430 vs 1826
winterthur → st-gallen | Same OD car allocation on both city pages | 214 vs 271
winterthur → schaffhausen | Same OD car allocation on both city pages | 121 vs 165
lucerne → zurich | Same OD car allocation on both city pages | 500 vs 288
st-gallen → winterthur | Same OD car allocation on both city pages | 163 vs 129
biel-bienne → bern | Same OD car allocation on both city pages

### 2. Inspect the selected city's denominators and transport coverage
Coverage percentages in this table use mapped people as the denominator; `localCoveragePct` in the full export instead uses eligible short-distance active estimates.

In [4]:
print("Population groups")
for r in audit["groups"]:
    if r["city"] == CITY:
        pprint({k:r[k] for k in ["group","summaryPeople","mappedPeople","modelPeople","carPct","transitPct","softPct"]})
print("Routing coverage")
for r in audit["modes"]:
    if r["city"] == CITY:
        pprint({k:r[k] for k in ["mode","direction","people","covered","coveragePct","eligiblePeople"]})

Population groups
{'carPct': 16.386161757830763,
 'group': 'foreignInbound',
 'mappedPeople': 4278,
 'modelPeople': 4278,
 'softPct': 21.73913043478261,
 'summaryPeople': 4278,
 'transitPct': 61.87470780738663}
{'carPct': 16.397266195055355,
 'group': 'swissInbound',
 'mappedPeople': 233667,
 'modelPeople': 244783,
 'softPct': 21.726217223655887,
 'summaryPeople': 244783,
 'transitPct': 61.876516581288755}
{'carPct': 16.396974099494013,
 'group': 'swissOutbound',
 'mappedPeople': 59883,
 'modelPeople': 65738,
 'softPct': 21.72736836831822,
 'summaryPeople': 65738,
 'transitPct': 61.875657532187766}
Routing coverage
{'coveragePct': 95.00205044084478,
 'covered': 37066,
 'direction': 'inbound',
 'eligiblePeople': 0,
 'mode': 'car',
 'people': 39016}
{'coveragePct': 77.49538143881765,
 'covered': 114098,
 'direction': 'inbound',
 'eligiblePeople': 0,
 'mode': 'transit',
 'people': 147232}
{'coveragePct': 73.15318103565004,
 'covered': 37818,
 'direction': 'inbound',
 'eligiblePeople': 401

### 3. Inspect missing routes and long active estimates
These are estimated people missing from the map, not observed unmet travel demand. The top ten are ranked by current model weight.

In [5]:
city_routes = [r for r in audit["routes"] if r["city"] == CITY]
missing = sorted((r for r in city_routes if r["geometry"] == "missing"), key=lambda r:r["commuters"], reverse=True)
for r in missing[:10]:
    print(r["mode"], r["originName"], "→", r["targetName"], r["commuters"])
long_active = [r for r in city_routes if r["mode"] == "soft" and r["directKm"] > 25]
print("Active estimates >25 km:", sum(r["commuters"] for r in long_active), "people across", len(long_active), "city-page rows")
print("Geometry flags:")
pprint([r for r in city_routes if r["geometryErrors"]][:5])

transit Volketswil → Zürich 1398
transit Fällanden → Zürich 1194
transit Oberengstringen → Zürich 1051
transit Herrliberg → Zürich 894
transit Konstanz → Zürich 813
transit Wettswil am Albis → Zürich 796
transit Spreitenbach → Zürich 787
transit Wangen-Brüttisellen → Zürich 786
transit Gossau (ZH) → Zürich 676
transit Waldshut-Tiengen → Zürich 665
Active estimates >25 km: 15016 people across 698 city-page rows
Geometry flags:
[{'activeEligible': False,
  'city': 'zurich',
  'commuters': 18,
  'directKm': 64.4911486289634,
  'direction': 'inbound',
  'geometry': 'road',
  'geometryErrors': 'endpoint snap exceeds review threshold',
  'group': 'swissInbound',
  'key': 'zurich:CH1631>CH261:car:inbound',
  'mode': 'car',
  'origin': 'CH1631',
  'originName': 'Glarus Süd',
  'routeKm': 90.30937924720517,
  'seconds': 8821.309,
  'skippedReason': '',
  'snapKm': 1.6351409726853414,
  'source': 'FSO commune matrix 2020',
  'target': 'CH261',
  'targetName': 'Zürich'}]


### 4. Compare the same OD pair across city pages
Both pages agree on the all-mode total for all 14 shared directed pairs. Mode discrepancies are created by the city-specific allocation.

In [6]:
pairs = [r for r in audit["reciprocal"] if r["mode"] == "car" and CITY in [r["origin"],r["target"]]]
pprint(pairs)
assert all(r["delta"] == 0 for r in audit["reciprocal"] if r["mode"] == "all")

[{'delta': 487,
  'mode': 'car',
  'origin': 'zurich',
  'originPage': 553,
  'target': 'winterthur',
  'targetPage': 1040},
 {'delta': 154,
  'mode': 'car',
  'origin': 'zurich',
  'originPage': 209,
  'target': 'lucerne',
  'targetPage': 363},
 {'delta': -1604,
  'mode': 'car',
  'origin': 'winterthur',
  'originPage': 3430,
  'target': 'zurich',
  'targetPage': 1826},
 {'delta': -212,
  'mode': 'car',
  'origin': 'lucerne',
  'originPage': 500,
  'target': 'zurich',
  'targetPage': 288}]


### 5. Inspect the reconstructed source categories
INSEE's TRANS 1 means no transport. It is currently included in `soft`. TRANS 4 means motorcycles; it is currently included in `car`. Source estimates are survey-weighted floats; the application rounds once per origin/category.

In [7]:
french = audit["french"]
print("French rows reproduced:", french["comparedRows"], "Mismatches:", french["mismatches"])
print("Unrounded sum:", french["rawSum"], "Rounded app total:", french["roundedSum"])
labels = {"1":"No transport","2":"Walking","3":"Bicycle/e-bike","4":"Motorcycle","5":"Car/van/truck","6":"Public transport"}
for mode, value in french["byOriginalMode"].items(): print(mode, labels[mode], round(value,2))
print("FSO mode population definition:", audit["originalSources"]["modeDefinition"]["fr"])
assert french["mismatches"] == 0 and not french["omitted"]

French rows reproduced: 714 Mismatches: 0
Unrounded sum: 119005.63183290715 Rounded app total: 119003
1 No transport 157.64
2 Walking 573.7
3 Bicycle/e-bike 6216.43
4 Motorcycle 6655.29
5 Car/van/truck 86598.5
6 Public transport 18804.08
FSO mode population definition: Pendulaires (sortants et intracommunaux)


### 6. Test sensitivity without calling it a correction
The percentages below are scenarios. They are not confidence bounds, current observations or proposed replacements. MIV includes motorcycles.

In [8]:
group = next(r for r in audit["groups"] if r["city"] == CITY and r["group"] == "swissInbound")
print("Mapped Swiss incoming workers:", group["mappedPeople"])
for share in sorted(set([group["carPct"]/100, .20, .25, .28, .30])):
    print(f"{share:.1%} → {round(group['mappedPeople']*share):,} motorised-private commuters")
print("Selected moments: minute, travelling by car, independent curve, journey-time curve")
for r in audit["hourly"]:
    if r["city"] == CITY and r["minute"] in [360,420,465,480,720,1050,1200]:
        print(r["minute"], r["carNow"], r["curveCarPopulationSameCoverage"], r["routeDerivedCarPopulation"])

Mapped Swiss incoming workers: 233667
16.4% → 38,315 motorised-private commuters
20.0% → 46,733 motorised-private commuters
25.0% → 58,417 motorised-private commuters
28.0% → 65,427 motorised-private commuters
30.0% → 70,100 motorised-private commuters
Selected moments: minute, travelling by car, independent curve, journey-time curve
360 2787 3324 -20
420 6730 8724 3454
465 9228 14810 9060
480 8871 16886 11696
720 0 27509 27733
1050 8889 12643 12243
1200 1212 1519 -196


## Takeaways
Fix cohort and category definitions before increasing route coverage. Use the same mode allocation for an OD pair on every page; exclude no-transport records; reconcile Geneva's canton groups. Then expose missing-route coverage and unify the population and journey clocks.

All row-level evidence is available in the CSV files beside this notebook. See `README.md` for source URLs, exact selections, validation limits and commands. The HTML report contains the ranked recommendations and three source-backed charts.